# 5. Equation discovery 

Let's have a coupled two-mass damped oscillator (a massive body mass coupled to a light tissue layer).

Your task is to discover the underlying governing physical equations directly from time-series observations.




In [ ]:
# import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso

Define parameters and generate observation data:

In [ ]:
# Data preparation
w1 = 0.182873627
w2 = 0.903787671

T_max = 8 * np.pi / w1
dt = 0.02
t = np.arange(0, T_max, dt)
N = len(t)

e1 = np.exp(-0.005584807 * t)
e2 = np.exp(-0.038415193 * t)
arg1 = w1 * t
arg2 = w2 * t

x1 = e1 * (1.00643755 * np.cos(arg1) + 0.03253009 * np.sin(arg1)) + \
     e2 * (-0.00643755 * np.cos(arg2) - 0.00063669 * np.sin(arg2))

x2 = e1 * (0.96520193 * np.cos(arg1) + 0.04741893 * np.sin(arg1)) + \
     e2 * (0.13479807 * np.cos(arg2) + 0.00209904 * np.sin(arg2))

v1 = e1 * ((-0.005584807 * 1.00643755 + 0.182873627 * 0.03253009) * np.cos(arg1) + 
           (-0.005584807 * 0.03253009 - 0.182873627 * 1.00643755) * np.sin(arg1)) + \
     e2 * ((-0.038415193 * (-0.00643755) + 0.903787671 * (-0.00063669)) * np.cos(arg2) + 
           (-0.038415193 * (-0.00063669) - 0.903787671 * (-0.00643755)) * np.sin(arg2))

v2 = e1 * ((-0.005584807 * 0.96520193 + 0.182873627 * 0.04741893) * np.cos(arg1) + 
           (-0.005584807 * 0.04741893 - 0.182873627 * 0.96520193) * np.sin(arg1)) + \
     e2 * ((-0.038415193 * 0.13479807 + 0.903787671 * 0.00209904) * np.cos(arg2) + 
           (-0.038415193 * 0.00209904 - 0.903787671 * 0.13479807) * np.sin(arg2))

# Plot of the time response
plt.figure(figsize=(12, 5))

# Displacements
plt.subplot(1, 2, 1)
plt.plot(t, x1, label='$x_1(t)$', lw=1.8)
plt.plot(t, x2, label='$x_2(t)$', lw=1.8, linestyle='--')
plt.title("Displacement Time History $x_1, x_2$")
plt.xlabel("Time $t$ [s]")
plt.ylabel("Displacement")
plt.grid(True, alpha=0.3)
plt.legend()

# Velocities
plt.subplot(1, 2, 2)
plt.plot(t, v1, label='$v_1(t)$', lw=1.5)
plt.plot(t, v2, label='$v_2(t)$', lw=1.5, linestyle='--')
plt.title("Velocity Time History $v_1, v_2$")
plt.xlabel("Time $t$ [s]")
plt.ylabel("Velocity")
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

Sparse regression using Lasso with a custom regularization parameter:

In [ ]:
alpha_user = 1e2  # Modify this value


feature_names = [
    'x1', 'v1', 'x2', 'v2',
    '1', 'x1v1'
]

# Candidate libraby - you can add some combinations
Theta = np.column_stack([
    x1, v1, x2, v2,
    np.ones_like(x1), x1*v1
])




a1 = np.gradient(v1, dt)
a2 = np.gradient(v2, dt)

X = np.column_stack([x1, v1, x2, v2])
dX = np.column_stack([v1, a1, v2, a2])

Xi_manual = np.zeros((Theta.shape[1], dX.shape[1]))

for j in range(dX.shape[1]):
    lasso = Lasso(alpha=alpha_user, fit_intercept=False, max_iter=40000, tol=1e-6)
    lasso.fit(Theta, dX[:, j])
    Xi_manual[:, j] = lasso.coef_


# Display reconstructed matrix
row_labels = ['dx1/dt', 'dv1/dt', 'dx2/dt', 'dv2/dt']
print(f"Reconstructed Matrix with alpha = {alpha_user:.1e}:\n")
header = f"{'':>8}  " + " ".join([f"{name:>9}" for name in feature_names])
print(header)

Xi_all = Xi_manual.T
for i, name in enumerate(row_labels):
    row_str = " ".join([f"{val:9.4f}" for val in Xi_all[i]])
    print(f"{name:>8}  {row_str}")


Manually guessing the right $\alpha$ is difficult due to shrinkage bias and scale imbalance. 

To automate the parameter selection and eliminate estimation bias, we combine **Post-Lasso** (feature selection via Lasso followed by an unpenalized OLS refit) with the **Bayesian Information Criterion (BIC)**:

In [ ]:
a1 = np.gradient(v1, dt)
a2 = np.gradient(v2, dt)

X = np.column_stack([x1, v1, x2, v2])
dX = np.column_stack([v1, a1, v2, a2])

feature_names = [
    'x1', 'v1', 'x2', 'v2', 
    '1', 
    'x1^2', 'x2^2', 'x1*x2', 
    'v1^2', 'v2^2', 'x1*v1', 'x2*v2'
]

Theta = np.column_stack([
    x1, v1, x2, v2,
    np.ones_like(x1),
    x1**2, x2**2, x1 * x2,
    v1**2, v2**2, x1 * v1, x2 * v2
])


# Post-Lasso sparse regression with BIC model selection
alphas_grid = np.logspace(-7, -2, 60)
Xi_post_lasso = np.zeros((Theta.shape[1], dX.shape[1]))
optimal_alphas = []
bic_history_dv1 = []

for j in range(dX.shape[1]):
    y = dX[:, j]
    best_bic = np.inf
    best_coef = np.zeros(Theta.shape[1])
    best_alpha = None

    # Feature selection 
    for alpha in alphas_grid:
        lasso = Lasso(alpha=alpha, fit_intercept=False, max_iter=20000, tol=1e-6)
        lasso.fit(Theta, y)
        active_idx = np.abs(lasso.coef_) > 1e-6
        k = np.sum(active_idx)
        
        if k == 0:
            if j == 1:
                bic_history_dv1.append(np.nan)
            continue

        # Post-lasso    
        coef_refit = np.zeros(Theta.shape[1])
        ols_solution = np.linalg.lstsq(Theta[:, active_idx], y, rcond=None)[0]
        coef_refit[active_idx] = ols_solution
        
        # Compute BIC
        residual = y - Theta[:, active_idx] @ ols_solution
        rss = np.sum(residual**2)
        bic = N * np.log(rss / N) + k * np.log(N)

        if j == 1:
            bic_history_dv1.append(bic)
        
        if bic < best_bic:
            best_bic = bic
            best_coef = coef_refit
            best_alpha = alpha
            
    Xi_post_lasso[:, j] = best_coef
    optimal_alphas.append(best_alpha)

# Vizualization of BIC
plt.figure(figsize=(7, 3.5))
plt.semilogx(alphas_grid, bic_history_dv1, 'b.-', lw=1.5, label='BIC score')
plt.axvline(optimal_alphas[1], color='r', linestyle='--', label=f'Optimal alpha = {optimal_alphas[1]:.1e}')
plt.title("Model Selection")
plt.xlabel(r"Regularisation parameter $\alpha$")
plt.ylabel("BIC")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Result matrix
row_labels = ['dx1/dt', 'dv1/dt', 'dx2/dt', 'dv2/dt']

header = f"{'':>8}  " + " ".join([f"{name:>9}" for name in feature_names])
print(header)

Xi_all = Xi_post_lasso.T
for i, name in enumerate(row_labels):
    row_str = " ".join([f"{val:9.4f}" for val in Xi_all[i]])
    print(f"{name:>8}  {row_str}")

print("\n\n")
print("Final equations:")
for i, eq_label in enumerate(row_labels):
    terms = []
    for coef, feat in zip(Xi_all[i], feature_names):
        if np.abs(coef) > 1e-4:
            feat_str = "" if feat == '1' else f" * {feat}"
            terms.append((coef, feat_str))
    
    if not terms:
        eq_str = "0.0000"
    else:
        first_coef, first_feat = terms[0]
        eq_str = f"{first_coef:.4f}{first_feat}"

        for coef, feat in terms[1:]:
            sign = " - " if coef < 0 else " + "
            eq_str += f"{sign}{abs(coef):.4f}{feat}"
            
    print(f"  {eq_label:>8} = {eq_str}")
